### Learning Goal
- **What concept is this notebook teaching?** Text Clustering & Dimensionality Reduction (k-Means & PCA on NLP data).
- **Why does it matter?** It demonstrates how to apply Unsupervised Learning to unstructured text data. By converting BBC news articles into numbers (TF-IDF), we can mathematically group similar articles into topics without any human labeling.

In [ ]:
import pandas as pd

df = pd.read_csv('./data/bbc_news.csv')
print(df.head())
print(df['category'].value_counts())

### Experiment
- **What are we changing?** We are converting raw text into numerical vectors using `TfidfVectorizer`, and then feeding those high-dimensional vectors into a `KMeans` clustering algorithm to group them into 5 distinct news topics.
- **What do we expect to happen?** The TF-IDF transformation will give higher mathematical weight to rare, topic-specific words (like 'economy' or 'olympics') and ignore common filler words (like 'the' or 'and'), allowing k-Means to cleanly cluster the articles.

### Observation
- **What actually happened?** The `TfidfVectorizer` successfully mapped thousands of text articles into a sparse numerical matrix. The k-Means algorithm was then able to calculate Euclidean distance across thousands of word-dimensions to find the optimal cluster centroids.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['news'])

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

wcss = []
k_range = range(1, 10)

for i in k_range:
    # Partition unlabeled data into k clusters by iteratively minimizing the within-cluster sum of squares (WCSS).
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init='auto')
    kmeans.fit(X_tfidf)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o')
plt.title('Elbow Method for Topic Grouping', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters', fontsize=12)
plt.ylabel('WCSS', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization Analysis: Elbow Curve
- **Marking Values:** Look for the "elbow" or "hinge" point on the X-axis where the line sharply changes trajectory and begins flattening out.
- **Correct Interpretation:** This point represents the optimal number of clusters (k). Adding more clusters beyond this point yields rapidly diminishing returns in reducing variance.
- **How to Interpret:** The Y-axis (WCSS) measures how spread out the clusters are. We want tight clusters, but we don't want k to equal the number of data points.
- **Common Mistakes:** Blindly trusting the elbow curve when it looks more like a smooth slope. It is subjective and should be paired with the Silhouette Score.

In [ ]:
optimal_k = 5

# Partition unlabeled data into k clusters by iteratively minimizing the within-cluster sum of squares (WCSS).
kmeans_optimal = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init='auto')
y_kmeans = kmeans_optimal.fit_predict(X_tfidf)

df['cluster'] = y_kmeans

# Grouping without labels
cluster_summary = pd.DataFrame(df.groupby(['cluster', 'category']).size().unstack(fill_value=0))
print("Cluster to Actual Category Mapping (for evaluation):")
print(cluster_summary)

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

# Project high-dimensional data onto orthogonal axes of maximum variance to reduce dimensionality while preserving underlying structure.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_tfidf.toarray())

plt.figure(figsize=(10, 8))

colors = ['red', 'blue', 'green', 'cyan', 'magenta']
for i in range(optimal_k):
    plt.scatter(
        X_pca[y_kmeans == i, 0], 
        X_pca[y_kmeans == i, 1], 
        s=50, 
        c=colors[i], 
        label=f'Topic Cluster {i}',
        alpha=0.6,
        edgecolors='black'
    )

plt.title('News Articles Grouped by Topic (PCA 2D)', fontsize=15, fontweight='bold')
plt.xlabel('PCA Component 1', fontsize=12)
plt.ylabel('PCA Component 2', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization Analysis: Scatter/Pair Plot
- **Marking Values:** Look for distinct clusters, straight lines, or chaotic clouds of points.
- **Correct Interpretation:** A clear diagonal trend indicates a linear relationship. Distinct, separated blobs indicate the data is highly clusterable or easily classified.
- **How to Interpret:** Each dot represents a single row of data plotted across two feature dimensions.
- **Common Mistakes:** Over-interpreting a 2D scatter plot when the model actually operates in high-dimensional space (PCA should be used to crush dimensions first).

### Comparison
- **Clustering in High vs Low Dimensions:** We used PCA to crush the massive TF-IDF matrix (thousands of dimensions/words) down into just 2 Principal Components. The resulting scatter plot shows how cleanly the algorithm separated Business, Sports, Politics, Tech, and Entertainment articles, with only minor overlapping at the borders.

### Practical Takeaway
- **Industry Application:** Automated Document Categorization and Topic Modeling. Companies use these exact techniques to automatically organize massive archives of legal documents, support tickets, or daily news feeds without requiring humans to read and tag them.

### Key Insight
- **Memorable lesson:** PCA isn't just for numerical datasets. When dealing with text (NLP), it's the absolute best way to visualize thousands of unique words on a simple 2D screen to prove your clustering algorithm actually worked.